# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Runs the same transform as `scripts/01_prepare_features.py`: log-scale the traffic volume columns, keep the pre-built tier categoricals, and fill missing numerics with 0 rather than drop rows (missingness itself is informative — a page with 0 impressions is a real state, not noise).

In [1]:
import sys, subprocess
sys.path.insert(0, "../../scripts")

subprocess.run([sys.executable, "../../scripts/01_prepare_features.py"], check=True)

import pandas as pd
frame = pd.read_csv("../../data/processed/refresh_feature_vector.csv")
print(f"Built feature vector: {frame.shape[0]:,} rows x {frame.shape[1]} columns")
frame.filter(like="log_").head(3)


Prepared 30,000 rows from 30,000 raw rows
Wrote /home/claude/repo/data/processed/refresh_feature_vector.csv


Built feature vector: 30,000 rows x 52 columns


,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d
0,8.243808,3.401197,2.890372,0.0
1,9.636980,2.079442,2.302585,0.0
2,9.440023,2.484907,2.484907,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| `log_impressions_90d` | log1p of 90-day search impressions | fill 0 | Yes — trailing window |
| `avg_position` | average SERP position over 90d | fill 0 (treated as 'no ranked position') | Yes |
| `days_with_impressions` | how many of the last 90 days had any impressions | fill 0 | Yes |
| `content_age_days` | age of the page | never missing | Yes |
| `days_since_last_update` | staleness signal | never missing | Yes |
| `word_count`, `char_count` | on-page depth | fill 0 | Yes — static page property |
| `ctr`, `engagement_rate`, `scroll_rate` | engagement quality, 90d | fill 0 | Yes |
| `content_type`, `main_intent`, tier fields | categorical context | fill `"unknown"` | Yes — static |

All model features describe the page's state over the trailing 90-day window **up to** the scoring moment — none of them look inside the 30-day windows the label itself is built from (see leakage hunt below).

In [2]:
numeric_missing = frame.filter(regex="log_|days_|ctr|position|rate|count").isna().sum()
print("Missing values before fill (should be handled upstream by the pipeline):")
print(numeric_missing[numeric_missing > 0] if numeric_missing.sum() else "None — all filled at build time")


Missing values before fill (should be handled upstream by the pipeline):
None — all filled at build time


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The label (`is_declining_label`) is built from `trend_direction`, which compares `impressions_last_30d` to `impressions_prev_30d`. If `impressions_90d` — a feature I do use — is really just those same two windows in disguise, the model would be reading the label off the input.

In [3]:
overlap = frame[["impressions_90d", "impressions_last_30d", "impressions_prev_30d"]].copy()
overlap["last60_sum"] = overlap["impressions_last_30d"] + overlap["impressions_prev_30d"]

corr = overlap["impressions_90d"].corr(overlap["last60_sum"])
coverage = (overlap["last60_sum"] / overlap["impressions_90d"].replace(0, float("nan"))).mean()

print(f"Correlation(impressions_90d, last60_sum): {corr:.2f}")
print(f"Average share of the 90d total covered by the labels 60d window: {coverage:.1%}")
print()
print("Verdict: impressions_90d is closely related to the label windows (0.98 correlation),")
print("so the RAW last_30d / prev_30d windows and trend_direction / trend_pct themselves")
print("are kept OUT of the model feature list entirely — see exclusions below.")


Correlation(impressions_90d, last60_sum): 0.98
Average share of the 90d total covered by the labels 60d window: 56.2%

Verdict: impressions_90d is closely related to the label windows (0.98 correlation),
so the RAW last_30d / prev_30d windows and trend_direction / trend_pct themselves
are kept OUT of the model feature list entirely — see exclusions below.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`trend_direction`, `trend_pct`** — define the label itself; using either is reading the answer off the input.
- **`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`** — the raw windows the label is computed from; kept out even though `impressions_90d` (a coarser aggregate) is allowed in.
- **`provider_used`, `model_used`** — internal tooling/authorship metadata, not a content or search-visibility signal; using them risks the model learning "which tool wrote this" instead of "is this page declining."
- **`client_id`** — used only to group the train/test split, never fed to the model as a feature, so the model can't learn client identity as a shortcut.
- **Titles, URLs, domains, keywords** — not present in this anonymized release at all; excluded at the data-safety level, before feature engineering even starts.

In [4]:
import sys; sys.path.insert(0, "../../scripts")
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

excluded = ["trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d",
            "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
            "provider_used", "model_used", "client_id"]
leaked = [f for f in excluded if f in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES]
print("Excluded fields that leaked into the model feature list:", leaked or "none")


Excluded fields that leaked into the model feature list: none


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.